In [29]:
import pandas as pd
import duckdb
import os
import glob

In [30]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

In [31]:
# ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
# ...OR VICE VERSA?

# THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
# THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
# OR THEY ARE A COMINBATION OF MULTIPLE child PARTS.
# EITHER WAY, THE child(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

# TABLES NEEDED (6 out of 12):

    # part_relationships - core table to identify child-child connections
    # parts - to get the part name, for human readability
    # part_categories - allow for more granular analysis by part category
    # inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


#---note: the 'inventory_sets' table is not needed as it just records the quantity of a given part used per set, which is not needed here

In [32]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
inventory_sets = dfs["inventory_sets"]
sets = dfs["sets"]

In [33]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [64]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
joined = duckdb.sql("""
                    
                    WITH parts_sets AS(
                        SELECT
                            p.part_num,
                            s.set_num,
                            s.year,
                            ivs.quantity
                        FROM parts p JOIN inventory_parts ip ON p.part_num = ip.part_num
                        JOIN inventories i ON ip.inventory_id = i.id
                        JOIN inventory_sets ivs ON i.set_num = ivs.set_num
                        JOIN sets s ON ivs.set_num = s.set_num
                    )

                    SELECT DISTINCT ON(pr.parent_part_num, pr.child_part_num)
                        
                        pc.name AS category,                         --only needs to be shows once, as parent and child likely to have same category 
                    
                        --parent_ps.quantity AS parent_quantity,
                        --parent_ps.year AS parent_year,
                        --parent_ps.set_num AS parent_set_sum,
                        pr.parent_part_num,
                        parent_p.name AS parent_part_name,
                    
                        child_p.name AS child_part_name,
                        pr.child_part_num
                        --child_ps.set_num AS child_set_num,
                        --child_ps.year AS child_year,
                        --child_ps.quantity AS child_quantity
                    
                    FROM part_relationships pr
                    JOIN parts child_p ON pr.child_part_num = child_p.part_num
                    JOIN parts parent_p ON pr.parent_part_num = parent_p.part_num
                    JOIN part_categories pc ON child_p.part_cat_id = pc.id

                    JOIN parts_sets parent_ps ON parent_p.part_num = parent_ps.part_num
                    JOIN parts_sets child_ps ON child_p.part_num = child_ps.part_num

                    WHERE pr.rel_type = 'R'

                    ORDER BY pc.name
                    


                    
                    """).df() #takes 3.2 - 3.3 seconds to executed

In [65]:
joined


,category,parent_part_num,parent_part_name,child_part_name,child_part_num
0,Animal / Creature Body Parts,50108pr0001,"Creature Body Part, Dragon Head Upper Jaw with...","Creature Body Part, Dragon Head Lower Jaw with...",59227pr0001
1,Animal / Creature Body Parts,43746pr0001,"Animal Body Part, Snake / Serpent Head, Basili...","Animal Body Part, Snake / Serpent Neck S-Curve...",43750pr0001
2,Animal / Creature Body Parts,6127,"Animal / Creature Body Part, Dinosaur / Dragon...","Creature Body Part, Dragon Body Classic with O...",6129c01pr0001
3,Animal / Creature Body Parts,40245,"Creature Body Part, Dog Body, Three Heads (Flu...","Animal Body Part, Dog Head (Fluffy)",40246
4,Animal / Creature Body Parts,40396,"Animal / Creature Body Part, Neck / Tail, S Cu...","Animal Body Part, Elephant Head with Pin",43890c01
...,...,...,...,...,...
973,Windscreens and Fuselage,18907,Aircraft Fuselage Curved Forward 6 x 10 Top wi...,Glass for Aircraft Fuselage Curved Forward 6 x...,18908
974,Windscreens and Fuselage,4214,Hinge Vehicle Roof Holder 1 x 4 x 2,Windscreen 6 x 4 x 2 Canopy,4474
975,Windscreens and Fuselage,4625,Hinge Tile 1 x 4,Windscreen 1 x 4 x 1 1/3 with Bottom Hinge,30161
976,Windscreens and Fuselage,2917,Train Front Sloping Top,Glass for Train Front Sloping Top,2918


In [56]:
joined_cats = duckdb.sql("SELECT category, COUNT(category) FROM joined GROUP BY category ORDER BY category")
joined_cats

┌─────────────────────────────────────────┬─────────────────┐
│                category                 │ count(category) │
│                 varchar                 │      int64      │
├─────────────────────────────────────────┼─────────────────┤
│ Animal / Creature Body Parts            │            1747 │
│ Animals / Creatures                     │             420 │
│ Bars, Ladders and Fences                │            2847 │
│ Belville, Scala and Fabuland            │              26 │
│ Bricks                                  │               1 │
│ Bricks Curved                           │           12183 │
│ Bricks Round and Cones                  │              42 │
│ Bricks Sloped                           │               9 │
│ Bricks Special                          │             634 │
│ Bricks Wedged                           │           32931 │
│      ·                                  │             ·   │
│      ·                                  │             ·   │
│      ·

In [58]:
cats = duckdb.sql("""
                SELECT
                    c.name, 
                    COUNT(p.part_num)
                FROM part_categories c JOIN parts p ON p.part_cat_id = c.id
                WHERE c.name IN(SELECT DISTINCT category FROM joined)
                GROUP BY c.name
                ORDER BY c.name
                  """)
cats

┌─────────────────────────────────────────┬───────────────────┐
│                  name                   │ count(p.part_num) │
│                 varchar                 │       int64       │
├─────────────────────────────────────────┼───────────────────┤
│ Animal / Creature Body Parts            │               936 │
│ Animals / Creatures                     │               870 │
│ Bars, Ladders and Fences                │               136 │
│ Belville, Scala and Fabuland            │               742 │
│ Bricks                                  │              1756 │
│ Bricks Curved                           │               974 │
│ Bricks Round and Cones                  │               598 │
│ Bricks Sloped                           │               626 │
│ Bricks Special                          │               324 │
│ Bricks Wedged                           │               330 │
│      ·                                  │                 · │
│      ·                                